# Détection et segmentation d'instances avec YOLO

Dans ce TP, vous détectez les objets de photos de COCO avec YOLO26, un détecteur pré-entraîné d'Ultralytics, et vous mesurez sa qualité comme le font les articles : par la mAP, sur des images que le modèle n'a jamais vues. Vous comparez ses deux têtes, avec et sans NMS, faites correspondre ses classes aux catégories de COCO, évaluez tout l'échantillon, avec sa courbe précision-rappel, par classe et par taille d'objet, puis comparez la vitesse et la précision de plusieurs réglages. Vous passez enfin à la segmentation d'instances, avec YOLO26-seg puis Mask R-CNN, et évaluez leurs masques.

Chaque exercice s'appuie sur le précédent : faites-les dans l'ordre. Le TP tourne sans GPU, en quelques minutes par passe sur l'échantillon ; avec un GPU (Exécution > Modifier le type d'exécution), chaque passe prend quelques secondes.

## Installation et données

Les photos sont 943 images de la partie validation de [COCO 2017](https://cocodataset.org), sous licence CC BY 2.0 ou CC BY-SA 2.0 (leurs auteurs sont dans `credits.csv`), avec leurs annotations (CC BY 4.0, COCO Consortium) : des boîtes, des masques et une catégorie parmi 80, au format COCO. Les modèles de ce TP ont été entraînés sur la partie entraînement de COCO : ils n'ont jamais vu ces photos. Le dossier `coco-val2017-open` de [shuuchuu/datasets](https://github.com/shuuchuu/datasets) fait 156 Mo.

Ultralytics et torchmetrics ne sont pas installés sur Colab. Les poids des modèles se téléchargent à leur premier chargement.

In [ ]:
!pip install -q ultralytics torchmetrics pycocotools
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh coco-val2017-open

In [ ]:
import contextlib
import io
import time

import matplotlib.pyplot as plt
import torch
from matplotlib import patches
from pycocotools.coco import COCO
from torchmetrics.detection import MeanAveragePrecision
from torchvision.io import decode_image
from torchvision.models.detection import (
  MaskRCNN_ResNet50_FPN_V2_Weights,
  maskrcnn_resnet50_fpn_v2,
)
from ultralytics import YOLO

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Calcul sur {DEVICE}")

`coco` lit les annotations avec l'API de COCO (`pycocotools`) : `coco.imgs[id]` décrit une image, `coco.getAnnIds(imgIds=id)` donne les identifiants de ses annotations et `coco.loadAnns(...)` les annotations elles-mêmes. `categories` associe à chaque identifiant de catégorie son nom. `chemin` donne le fichier d'une image, et `afficher` montre une image avec des boîtes, données comme `x1, y1, x2, y2` en pixels, et leurs catégories COCO.

In [ ]:
with contextlib.redirect_stdout(io.StringIO()):  # COCO affiche sa progression
  coco = COCO("coco-val2017-open/annotations.json")
ids_images = sorted(coco.getImgIds())
categories = {c["id"]: c["name"] for c in coco.loadCats(coco.getCatIds())}


def chemin(id_image):
  return f"coco-val2017-open/images/{coco.imgs[id_image]['file_name']}"


def afficher(id_image, boites, etiquettes, titre=None):
  image = decode_image(chemin(id_image)).permute(1, 2, 0)
  fig, ax = plt.subplots(figsize=(8, 6))
  ax.imshow(image)
  for (x1, y1, x2, y2), etiquette in zip(boites.tolist(), etiquettes.tolist()):
    ax.add_patch(
      patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1, fill=False, color="yellow", linewidth=2
      )
    )
    ax.text(x1, y1, categories[etiquette], color="black", backgroundcolor="yellow")
  ax.set_title(titre)
  ax.axis("off")
  plt.show()


ID = ids_images[1]
print(len(ids_images), "images, la deuxième :", coco.imgs[ID])

## 1. Les annotations COCO

Comptez les annotations de l'échantillon, et celles qui ont `iscrowd` à 1 : une seule boîte pour une foule d'objets qu'on ne sépare pas (des spectateurs, des livres en rayon). Combien y a-t-il de catégories, et quels sont le plus petit et le plus grand identifiant ?

Écrivez ensuite `cibles(id_image)`, qui renvoie les annotations d'une image dans le format qu'attend torchmetrics : un dictionnaire avec `"boxes"`, un tenseur `float32` de forme `(N, 4)` en `x1, y1, x2, y2`, `"labels"`, les identifiants de catégorie (`int64`), et `"iscrowd"` (`int64`). COCO donne chaque boîte (`"bbox"`) en `x, y, largeur, hauteur`. Affichez les annotations de `ID`.

**Documentation utile**

- https://github.com/cocodataset/cocoapi/blob/master/PythonAPI/pycocotools/coco.py

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `coco.anns` est le dictionnaire de toutes les annotations, `ann["iscrowd"]` vaut 0 ou 1.
- `x, y, larg, haut = ann["bbox"]` donne la boîte `[x, y, x + larg, y + haut]`.
- `torch.tensor(liste, dtype=torch.float32).reshape(-1, 4)` garde la forme `(0, 4)` pour une image sans annotation.
- Pour `ID`, `cibles(ID)["boxes"]` a 4 lignes.

```python
annotations = list(coco.anns.values())
print(len(annotations), sum(...))
print(len(categories), min(categories), max(categories))


def cibles(id_image):
  anns = coco.loadAnns(coco.getAnnIds(imgIds=id_image))
  boites = [
    [x, y, x + larg, y + haut] for x, y, larg, haut in (a["bbox"] for a in anns)
  ]
  return {
    "boxes": torch.tensor(boites, dtype=torch.float32).reshape(-1, 4),
    "labels": torch.tensor(..., dtype=torch.int64),
    "iscrowd": ...,
  }


cible = cibles(ID)
afficher(ID, cible["boxes"], cible["labels"], "Annotations")
```

### Solution

In [ ]:
annotations = list(coco.anns.values())
print(
  len(annotations), "annotations,", sum(a["iscrowd"] for a in annotations), "foules"
)
print(len(categories), "catégories, de", min(categories), "à", max(categories))


def cibles(id_image):
  anns = coco.loadAnns(coco.getAnnIds(imgIds=id_image))
  boites = [
    [x, y, x + larg, y + haut] for x, y, larg, haut in (a["bbox"] for a in anns)
  ]
  return {
    "boxes": torch.tensor(boites, dtype=torch.float32).reshape(-1, 4),
    "labels": torch.tensor([a["category_id"] for a in anns], dtype=torch.int64),
    "iscrowd": torch.tensor([a["iscrowd"] for a in anns], dtype=torch.int64),
  }


cible = cibles(ID)
afficher(ID, cible["boxes"], cible["labels"], "Annotations")
# 6 944 annotations, dont 77 foules. 80 catégories, numérotées de 1 à 90 : il y a des
# trous dans la numérotation (exercice 4).

## 2. Une première détection

Chargez YOLO26 dans sa plus petite taille, `yolo26n.pt`, dans `modele`, et prédisez les objets de l'image `ID` avec `predict`. Affichez les boîtes, les scores et les indices de classe du résultat, les noms de ces classes (`modele.names`) et le temps de calcul (`speed`, en millisecondes). Montrez l'image annotée par le résultat (`plot`).

Comparez avec les annotations de l'exercice 1 : quels objets le modèle trouve-t-il, lesquels manque-t-il ?

**Documentation utile**

- https://docs.ultralytics.com/modes/predict/#working-with-results
- https://docs.ultralytics.com/reference/engine/results/

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `modele.predict(fichier, device=DEVICE, verbose=False)` renvoie une liste de résultats, un par image : prenez `[0]`.
- `r.boxes.xyxy`, `r.boxes.conf` et `r.boxes.cls` sont des tenseurs ; `cls` contient des flottants, à convertir en `int` pour indexer `modele.names`.
- `r.plot()` renvoie l'image annotée en BGR (l'ordre d'OpenCV) : `[..., ::-1]` la remet en RGB pour `plt.imshow`.

```python
modele = YOLO(...)
r = modele.predict(chemin(ID), device=DEVICE, verbose=False)[0]
print(r.boxes.xyxy.round(), r.boxes.conf, r.boxes.cls)
print([modele.names[int(c)] for c in r.boxes.cls])
print(r.speed)
plt.imshow(r.plot()[..., ::-1])
plt.axis("off")
plt.show()
```

### Solution

In [ ]:
modele = YOLO("yolo26n.pt")
r = modele.predict(chemin(ID), device=DEVICE, verbose=False)[0]
print(r.boxes.xyxy.round(), r.boxes.conf, r.boxes.cls)
print([modele.names[int(c)] for c in r.boxes.cls])
print(r.speed)
plt.figure(figsize=(8, 6))
plt.imshow(r.plot()[..., ::-1])
plt.axis("off")
plt.show()
# Trois détections au-dessus du seuil par défaut (0,25) : les deux joueurs et le gant,
# avec des scores entre 0,55 et 0,6. La balle, annotée (19 × 17 pixels), n'est pas
# détectée. Le temps se partage entre la préparation de l'image, le réseau
# (inference) et le tri des boîtes (postprocess).

## 3. NMS ou tête un pour un

`predict` ne garde que les boîtes dont le score dépasse `conf` (0,25 par défaut), puis applique la NMS : parmi des boîtes de même classe qui se recouvrent avec une IoU au-dessus de `iou` (0,5 par défaut avec `predict`, 0,7 avec `val`), seule la meilleure reste. La tête qui produit ces boîtes, la tête un pour plusieurs, en propose plusieurs par objet : la NMS fait le tri. YOLO26 a une seconde tête, la tête un pour un, entraînée à ne prédire qu'une boîte par objet, qui se passe de NMS : `nms=False` la choisit, `nms=None` (le réglage par défaut) revient à la première.

Prédisez l'image `ID` avec chacune des deux têtes, à `conf=0.01` pour garder aussi les détections faibles. Comptez les boîtes de chaque tête, et celles dont le score atteint 0,25, puis affichez les deux images côte à côte. La tête un pour un ne donne-t-elle qu'une boîte par objet ? L'effet de `iou` sur la mAP se mesurera à l'exercice 8.

Attention : Ultralytics garde `nms` d'un appel à l'autre, tant qu'un appel ne le change pas. Finissez par un appel avec `nms=None`, sur lequel comptent les exercices suivants.

**Documentation utile**

- https://docs.ultralytics.com/modes/predict/#inference-arguments
- https://docs.ultralytics.com/models/yolo26/

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `len(r.boxes)` compte les boîtes d'un résultat, `(r.boxes.conf >= 0.25).sum()` celles dont le score atteint 0,25.
- `plt.subplots(1, 2, figsize=(16, 6))` donne deux graphiques côte à côte ; `r.plot(labels=False)` dessine les boîtes sans leurs étiquettes, plus lisibles quand elles sont nombreuses.

```python
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for nms, ax in zip((None, False), axes):
  r = modele.predict(chemin(ID), conf=..., nms=nms, device=DEVICE, verbose=False)[0]
  print(f"nms={nms} :", len(r.boxes), "boîtes, dont", int(...), "à 0,25 ou plus")
  ax.imshow(r.plot(labels=False)[..., ::-1])
  ax.set_title(f"nms={nms}")
  ax.axis("off")
modele.predict(chemin(ID), nms=..., device=DEVICE, verbose=False)
plt.show()
```

### Solution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for nms, ax in zip((None, False), axes):
  r = modele.predict(chemin(ID), conf=0.01, nms=nms, device=DEVICE, verbose=False)[0]
  nb_haut = int((r.boxes.conf >= 0.25).sum())
  print(f"nms={nms} :", len(r.boxes), "boîtes, dont", nb_haut, "à 0,25 ou plus")
  ax.imshow(r.plot(labels=False)[..., ::-1])
  ax.set_title(f"nms={nms}")
  ax.axis("off")
modele.predict(chemin(ID), nms=None, device=DEVICE, verbose=False)
plt.show()
# À 0,25 ou plus, les deux têtes donnent les trois mêmes boîtes : les deux joueurs et
# le gant. À conf=0.01, la NMS laisse 14 boîtes, la tête un pour un 20 : sans NMS,
# iou ne sert plus, et aucun tri ne retire de boîtes. La tête un pour un ne supprime
# pas les doublons, elle leur donne des scores bas : entraînée à ne donner un score
# haut qu'à une boîte par objet, elle laisse aux autres boîtes des joueurs et du gant
# moins de 0,07. Un seuil bas garde donc ces doublons, comme des détections faibles
# ailleurs (un second gant à 0,11, la balle à 0,01). Au seuil par défaut, chaque objet
# trouvé n'a qu'une boîte.

## 4. Des classes de YOLO aux catégories COCO

YOLO numérote ses 80 classes de 0 à 79, COCO ses catégories de 1 à 90. Construisez `vers_coco`, un tenseur de 80 entiers (`int64`) tel que `vers_coco[i]` soit l'identifiant COCO de la classe `i` de YOLO, en associant les noms. Pour combien de classes l'identifiant COCO n'est-il pas `i + 1` ? Affichez les identifiants de 1 à 90 qui ne sont pas des catégories.

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Inversez `categories` (nom → identifiant), puis parcourez `modele.names[i]` pour `i` de 0 à 79.
- Sans cette conversion, une « personne » prédite (classe 0) serait comparée aux annotations de la catégorie 0, qui n'existe pas, et une « voiture » (classe 2) à celles de la catégorie 2, les vélos.

```python
id_par_nom = {nom: id_coco for id_coco, nom in categories.items()}
vers_coco = torch.tensor([... for i in range(80)])
print(sum(...))
print([i for i in range(1, 91) if ...])
```

### Solution

In [ ]:
id_par_nom = {nom: id_coco for id_coco, nom in categories.items()}
vers_coco = torch.tensor([id_par_nom[modele.names[i]] for i in range(80)])
print(sum(int(vers_coco[i]) != i + 1 for i in range(80)), "classes décalées")
print("absents :", [i for i in range(1, 91) if i not in categories])
# 69 classes sur 80 sont décalées. L'article de COCO prévoyait 91 catégories ; onze
# n'ont pas été annotées (12, 26, 29, 30, 45, 66, 68, 69, 71, 83, et 91 au-delà de 90),
# et leurs numéros sont restés libres.

## 5. Prédire tout l'échantillon

Écrivez `predire(modele, id_image, **options)`, qui prédit une image et renvoie sa prédiction dans le format de torchmetrics, sur le CPU : `"boxes"` (`x1, y1, x2, y2`), `"scores"` et `"labels"`, les identifiants COCO (`vers_coco`). Les options passent telles quelles à `predict`.

Prédisez les 943 images avec `conf=0.001` et `max_det=100` dans le dictionnaire `predictions` (identifiant de l'image → prédiction), et mesurez le temps moyen par image. Un seuil si bas garde presque toutes les boîtes : l'évaluation de l'exercice suivant en a besoin, et un seuil plus haut s'appliquera ensuite aux scores, sans tout prédire à nouveau.

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Prédisez une image à la fois : une liste de chemins passée à `predict` est traitée en un seul lot, toutes les images en mémoire (près de 9 Go pour 300 photos).
- `vers_coco[r.boxes.cls.long().cpu()]` convertit d'un coup les classes d'une image.
- `.cpu()` ramène un tenseur du GPU : sur le GPU, `predictions` occuperait sa mémoire.
- `time.perf_counter()` avant et après la boucle donne la durée.

```python
def predire(modele, id_image, **options):
  r = modele.predict(chemin(id_image), device=DEVICE, verbose=False, **options)[0]
  return {
    "boxes": r.boxes.xyxy.cpu(),
    "scores": ...,
    "labels": vers_coco[...],
  }


debut = time.perf_counter()
predictions = {i: predire(modele, i, conf=..., max_det=...) for i in ids_images}
duree = time.perf_counter() - debut
print(f"{1000 * duree / len(ids_images):.0f} ms par image")
```

### Solution

In [ ]:
def predire(modele, id_image, **options):
  r = modele.predict(chemin(id_image), device=DEVICE, verbose=False, **options)[0]
  return {
    "boxes": r.boxes.xyxy.cpu(),
    "scores": r.boxes.conf.cpu(),
    "labels": vers_coco[r.boxes.cls.long().cpu()],
  }


debut = time.perf_counter()
predictions = {i: predire(modele, i, conf=0.001, max_det=100) for i in ids_images}
duree = time.perf_counter() - debut
print(f"{1000 * duree / len(ids_images):.0f} ms par image")
print(sum(len(p["scores"]) for p in predictions.values()), "boîtes")
# Sur les deux cœurs de Colab, de l'ordre de 100 à 200 ms par image ; quelques
# millisecondes sur un GPU, une fois le modèle chargé.

## 6. La mAP et la courbe précision-rappel

Écrivez `evaluer(predictions, seuil=0.0, **options)`, qui ne garde de chaque prédiction que les boîtes de score au moins égal à `seuil`, les passe avec les annotations (`cibles`) à un `MeanAveragePrecision` créé avec les options données, et renvoie le résultat de `compute`.

Évaluez `predictions` sans seuil, puis avec le seuil par défaut de `predict`, 0,25. Comparez `map` (la mAP de COCO, moyennée sur les IoU de 0,5 à 0,95), `map_50` et `map_75` : pourquoi le seuil de 0,25 fait-il baisser la mAP ? Lequel des deux réglages donne la mAP du modèle, lequel servirait dans une application ?

Tracez enfin la courbe précision-rappel de la catégorie personne (identifiant 1), aux IoU 0,5 et 0,75. Avec `extended_summary=True`, le résultat de `compute` a deux clés de plus, `precision` et `scores` : la précision et le score atteints à chacun des 101 niveaux de rappel (0, 0,01… 1), dans un tenseur de forme T × R × K × A × M (10 seuils d'IoU, de 0,5 à 0,95 ; 101 niveaux de rappel ; les catégories de `res["classes"]` ; 4 tailles d'objet, toutes d'abord ; 3 nombres maximaux de détections, 100 en dernier). Marquez sur chaque courbe le point où le score passe sous 0,25, et colorez l'aire qui le suit : que représente-t-elle ?

**Documentation utile**

- https://lightning.ai/docs/torchmetrics/stable/detection/mean_average_precision.html

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `metrique.update(liste_de_predictions, liste_de_cibles)` prend des listes d'images : `[p]`, `[cibles(i)]` pour une image.
- Un masque booléen `p["scores"] >= seuil` filtre les trois tenseurs d'une prédiction : `{cle: v[garde] for cle, v in p.items()}`.
- Les annotations `iscrowd` comptent comme dans COCO : une détection sur une foule n'est ni juste ni fausse.
- `res["precision"][t, :, k, 0, -1]` est la courbe du seuil d'IoU d'indice `t` (0 pour 0,5, 5 pour 0,75) et de la catégorie d'indice `k` dans `res["classes"]`, toutes tailles, 100 détections ; `res["scores"][t, :, k, 0, -1]` donne leurs scores, qui décroissent le long du rappel : `int((scores >= 0.25).sum()) - 1` est l'indice du dernier point au-dessus du seuil.
- `torch.linspace(0, 1, 101)` donne les niveaux de rappel ; `plt.fill_between(x, y, alpha=0.2)` colore l'aire sous une courbe.

```python
def evaluer(predictions, seuil=0.0, **options):
  metrique = MeanAveragePrecision(**options)
  for i, p in predictions.items():
    garde = ...
    metrique.update([{cle: v[garde] for cle, v in p.items()}], [...])
  return metrique.compute()


for seuil in (0.0, 0.25):
  res = evaluer(predictions, seuil)
  print(seuil, {cle: round(float(res[cle]), 3) for cle in ("map", "map_50", "map_75")})

res = evaluer(predictions, extended_summary=True)
k = res["classes"].tolist().index(...)
rappels = torch.linspace(0, 1, 101)
plt.figure(figsize=(8, 5))
for t, iou in ((0, 0.5), (5, 0.75)):
  precision = res["precision"][t, :, k, 0, -1]
  scores = ...
  j = ...
  courbe = plt.plot(rappels, precision, label=f"IoU {iou}")[0]
  plt.fill_between(rappels[j:], precision[j:], color=courbe.get_color(), alpha=0.2)
  plt.scatter(..., color=courbe.get_color(), zorder=3)
plt.xlabel("Rappel")
plt.ylabel("Précision")
plt.legend()
plt.show()
```

### Solution

In [ ]:
def evaluer(predictions, seuil=0.0, **options):
  metrique = MeanAveragePrecision(**options)
  for i, p in predictions.items():
    garde = p["scores"] >= seuil
    metrique.update([{cle: v[garde] for cle, v in p.items()}], [cibles(i)])
  return metrique.compute()


for seuil in (0.0, 0.25):
  res = evaluer(predictions, seuil)
  print(seuil, {cle: round(float(res[cle]), 3) for cle in ("map", "map_50", "map_75")})

res = evaluer(predictions, extended_summary=True)
k = res["classes"].tolist().index(1)  # la catégorie personne
rappels = torch.linspace(0, 1, 101)
plt.figure(figsize=(8, 5))
for t, iou in ((0, 0.5), (5, 0.75)):
  precision = res["precision"][t, :, k, 0, -1]
  scores = res["scores"][t, :, k, 0, -1]
  j = int((scores >= 0.25).sum()) - 1
  courbe = plt.plot(rappels, precision, label=f"IoU {iou}")[0]
  plt.fill_between(rappels[j:], precision[j:], color=courbe.get_color(), alpha=0.2)
  plt.scatter(rappels[j], precision[j], color=courbe.get_color(), zorder=3)
plt.xlabel("Rappel")
plt.ylabel("Précision")
plt.title("Personnes")
plt.legend()
plt.show()
# Sans seuil, mAP de 0,42 environ (0,59 à une IoU de 0,5, 0,44 à 0,75) ; avec le seuil
# de 0,25, 0,36. L'AP est l'aire sous la courbe précision-rappel, parcourue des scores
# les plus hauts aux plus bas : couper sous 0,25 arrête la courbe avant son rappel
# maximal, et l'aire qui manque est perdue. La mAP d'un modèle se mesure donc avec un
# seuil très bas ; une application choisit ensuite son seuil, d'après le compromis
# entre fausses détections et objets manqués qu'elle accepte.
# Sur la courbe des personnes, le point du seuil de 0,25 tombe avant la fin de la
# courbe : l'aire colorée qui le suit, ce sont les personnes retrouvées avec un score
# plus bas, à une précision plus basse. L'AP la compte, le seuil la perd. À une IoU de
# 0,75, la courbe est plus basse : une boîte doit être plus précise pour compter.

## 7. Par classe et par taille d'objet

Évaluez `predictions` sans seuil, avec `class_metrics=True`. Affichez la mAP des petits objets (moins de 32 × 32 pixels, `map_small`), des moyens et des grands (plus de 96 × 96 pixels), puis les cinq classes de plus faible et de plus forte AP parmi celles qui ont au moins 20 objets annotés (hors foules) dans l'échantillon, avec leur nombre d'objets. Qu'est-ce qui rend une classe difficile ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `res["classes"]` et `res["map_per_class"]` sont deux tenseurs alignés : `zip(res["classes"].tolist(), res["map_per_class"].tolist())`.
- `len(coco.getAnnIds(catIds=c, iscrowd=False))` compte les objets d'une catégorie.

```python
res = evaluer(predictions, class_metrics=True)
print(
  {cle: round(float(res[cle]), 3) for cle in ("map_small", "map_medium", "map_large")}
)
nombres = {c: len(coco.getAnnIds(catIds=c, iscrowd=False)) for c in categories}
par_classe = sorted(
  (ap, categories[c], nombres[c]) for c, ap in zip(...) if nombres[c] >= ...
)
print(par_classe[:5])
print(par_classe[-5:])
```

### Solution

In [ ]:
res = evaluer(predictions, class_metrics=True)
print(
  {cle: round(float(res[cle]), 3) for cle in ("map_small", "map_medium", "map_large")}
)
nombres = {c: len(coco.getAnnIds(catIds=c, iscrowd=False)) for c in categories}
par_classe = sorted(
  (round(ap, 3), categories[c], nombres[c])
  for c, ap in zip(res["classes"].tolist(), res["map_per_class"].tolist())
  if nombres[c] >= 20
)
print(par_classe[:5])
print(par_classe[-5:])
# Les petits objets ont une mAP près de quatre fois plus faible que les grands :
# quelques pixels sur la carte de caractéristiques au pas de 8, et une boîte décalée
# de deux pixels y perd beaucoup d'IoU. Les classes difficiles sont petites, fines ou
# en partie cachées (pommes, cuillères, couteaux, sacs à dos portés, battes de
# baseball) ; les plus faciles ont une forme typique, souvent sur un fond simple
# (chats, girafes, éléphants, souris d'ordinateur, frisbees).

## 8. Vitesse et précision

Sur les 300 premières images (`sous_ensemble`), comparez six réglages : YOLO26n en 640 pixels, avec la NMS à son IoU par défaut, 0,5 (le réglage de l'exercice 5, dont `predictions` a déjà les résultats), YOLO26n en 320 pixels (`imgsz=320`), YOLO26n avec la NMS à une IoU de 0,3 puis de 0,7 (`iou`), YOLO26n avec la tête un pour un (`nms=False`) et YOLO26s, la taille au-dessus (`yolo26s.pt`), en 640 pixels. Pour chacun, affichez le temps moyen par image, `map` et `map_small`. Que change l'IoU de la NMS ? Quel réglage choisiriez-vous pour une caméra embarquée, pour un traitement de photos par lots ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `{i: predictions[i] for i in sous_ensemble}` reprend les prédictions de l'exercice 5.
- Gardez `conf=0.001, max_det=100` pour chaque réglage, et mesurez la boucle de prédiction seule, pas l'évaluation.
- Le premier appel d'un modèle neuf est plus lent (chargement, préparation) : prédisez une image avant de mesurer.
- `nms=False` reste en place pour les appels suivants de `modele` (exercice 3) : gardez ce réglage pour la fin, et finissez par un appel avec `nms=None`.

```python
sous_ensemble = ids_images[:300]
modele_s = YOLO("yolo26s.pt")
reglages = {
  "n, 320 px": (modele, {"imgsz": 320}),
  "n, IoU 0,3": (modele, {"iou": 0.3}),
  "n, IoU 0,7": (modele, {"iou": 0.7}),
  "n, tête un pour un": (modele, {"nms": False}),
  "s, 640 px": (modele_s, {}),
}
mesures = {"n, 640 px": {i: predictions[i] for i in sous_ensemble}}
for nom, (m, options) in reglages.items():
  predire(m, sous_ensemble[0], **options)
  debut = time.perf_counter()
  mesures[nom] = {i: predire(m, i, conf=0.001, max_det=100, **options) for i in ...}
  print(nom, f"{1000 * (time.perf_counter() - debut) / len(sous_ensemble):.0f} ms")
modele.predict(chemin(ID), nms=..., device=DEVICE, verbose=False)
for nom, p in mesures.items():
  res = evaluer(p)
  print(nom, round(float(res["map"]), 3), round(float(res["map_small"]), 3))
```

### Solution

In [ ]:
sous_ensemble = ids_images[:300]
modele_s = YOLO("yolo26s.pt")
reglages = {
  "n, 320 px": (modele, {"imgsz": 320}),
  "n, IoU 0,3": (modele, {"iou": 0.3}),
  "n, IoU 0,7": (modele, {"iou": 0.7}),
  "n, tête un pour un": (modele, {"nms": False}),
  "s, 640 px": (modele_s, {}),
}
mesures = {"n, 640 px": {i: predictions[i] for i in sous_ensemble}}
for nom, (m, options) in reglages.items():
  predire(m, sous_ensemble[0], **options)
  debut = time.perf_counter()
  mesures[nom] = {
    i: predire(m, i, conf=0.001, max_det=100, **options) for i in sous_ensemble
  }
  print(nom, f"{1000 * (time.perf_counter() - debut) / len(sous_ensemble):.0f} ms")
modele.predict(chemin(ID), nms=None, device=DEVICE, verbose=False)
for nom, p in mesures.items():
  res = evaluer(p)
  print(nom, round(float(res["map"]), 3), round(float(res["map_small"]), 3))
# En 320 pixels, l'image a quatre fois moins de pixels : deux fois plus rapide, mais
# la mAP perd un quart, et celle des petits objets les trois quarts. Une IoU de 0,3
# supprime aussi des objets voisins de même classe qui se recouvrent (des joueurs, des
# voitures garées) : la mAP baisse. À 0,7, la NMS garde plus de boîtes, doublons
# compris, sans faire baisser la mAP : les doublons ont des scores plus bas que la
# bonne boîte, et l'AP ne les compte qu'en fin de courbe. D'où 0,7 pour val, qui
# mesure, et 0,5 pour predict, dont on regarde les boîtes. La tête un pour un
# fait aussi bien que la NMS, dans le même temps ici : la NMS coûte peu sur une
# image ; s'en passer simplifie surtout un modèle exporté (ONNX, TensorRT), qui n'a
# plus d'étape à part après le réseau. YOLO26s, quatre fois plus de paramètres, gagne
# six points de mAP pour un peu plus de deux fois plus de temps. Une caméra embarquée
# prendrait n (en 320 pixels si ses objets sont grands) ; un traitement par lots, sans
# contrainte de temps, s ou plus grand.

## 9. Segmentation d'instances

YOLO26-seg (`yolo26n-seg.pt`) ajoute à chaque boîte un masque, dans `r.masks.data` : un tenseur `(N, H, W)` de 0 et de 1, à la taille de l'image avec `retina_masks=True` (sinon à la taille d'entrée du réseau). `coco.annToMask(ann)` donne le masque d'une annotation, au même format.

Ajoutez à `cibles` et à `predire` un paramètre `masques=False` qui, à `True`, ajoute au dictionnaire une clé `"masks"` : un tenseur booléen `(N, H, W)`. Montrez les masques prédits sur l'image `ID`. Évaluez ensuite YOLO26n-seg sur `sous_ensemble`, sur les boîtes et sur les masques à la fois, avec `iou_type=("bbox", "segm")` : `MeanAveragePrecision` donne alors `bbox_map` et `segm_map`. Gardez les masques hors de `predictions` : mettez à jour la métrique image par image, au fil des prédictions.

La mAP des masques est-elle plus haute ou plus basse que celle des boîtes, et pourquoi ?

**Documentation utile**

- https://docs.ultralytics.com/tasks/segment/
- https://docs.ultralytics.com/reference/engine/results/#ultralytics.engine.results.Masks

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `torch.from_numpy(np_array)` convertit un masque de `annToMask` ; `torch.stack` les empile. Une image sans annotation garde la forme `(0, H, W)` : `torch.zeros((0, h, w), dtype=torch.bool)`, avec `h, w = coco.imgs[id_image]["height"], coco.imgs[id_image]["width"]`.
- Une image sans détection n'a pas de masques : `r.masks` vaut alors `None`.
- Prédire 300 images en 640 pixels, avec les masques, prend quelques minutes sans GPU.

```python
def cibles(id_image, masques=False):
  anns = coco.loadAnns(coco.getAnnIds(imgIds=id_image))
  boites = [
    [x, y, x + larg, y + haut] for x, y, larg, haut in (a["bbox"] for a in anns)
  ]
  cible = {
    "boxes": torch.tensor(boites, dtype=torch.float32).reshape(-1, 4),
    "labels": torch.tensor([a["category_id"] for a in anns], dtype=torch.int64),
    "iscrowd": torch.tensor([a["iscrowd"] for a in anns], dtype=torch.int64),
  }
  if masques:
    h, w = coco.imgs[id_image]["height"], coco.imgs[id_image]["width"]
    cible["masks"] = torch.zeros((0, h, w), dtype=torch.bool)
    if anns:
      cible["masks"] = torch.stack([...]).bool()
  return cible


def predire(modele, id_image, masques=False, **options):
  r = modele.predict(chemin(id_image), device=DEVICE, verbose=False, **options)[0]
  prediction = {...}
  if masques:
    prediction["masks"] = (
      r.masks.data.bool().cpu()
      if r.masks is not None
      else torch.zeros((0, *r.orig_shape), dtype=torch.bool)
    )
  return prediction


modele_seg = YOLO("yolo26n-seg.pt")
r = modele_seg.predict(chemin(ID), device=DEVICE, retina_masks=True, verbose=False)[0]
plt.imshow(r.plot()[..., ::-1])
plt.show()
metrique = MeanAveragePrecision(iou_type=("bbox", "segm"))
for i in sous_ensemble:
  p = predire(modele_seg, i, masques=True, conf=0.001, max_det=100, retina_masks=True)
  metrique.update(...)
res = metrique.compute()
print(...)
```

### Solution

In [ ]:
def cibles(id_image, masques=False):
  anns = coco.loadAnns(coco.getAnnIds(imgIds=id_image))
  boites = [
    [x, y, x + larg, y + haut] for x, y, larg, haut in (a["bbox"] for a in anns)
  ]
  cible = {
    "boxes": torch.tensor(boites, dtype=torch.float32).reshape(-1, 4),
    "labels": torch.tensor([a["category_id"] for a in anns], dtype=torch.int64),
    "iscrowd": torch.tensor([a["iscrowd"] for a in anns], dtype=torch.int64),
  }
  if masques:
    h, w = coco.imgs[id_image]["height"], coco.imgs[id_image]["width"]
    cible["masks"] = torch.zeros((0, h, w), dtype=torch.bool)
    if anns:
      cible["masks"] = torch.stack(
        [torch.from_numpy(coco.annToMask(a)) for a in anns]
      ).bool()
  return cible


def predire(modele, id_image, masques=False, **options):
  r = modele.predict(chemin(id_image), device=DEVICE, verbose=False, **options)[0]
  prediction = {
    "boxes": r.boxes.xyxy.cpu(),
    "scores": r.boxes.conf.cpu(),
    "labels": vers_coco[r.boxes.cls.long().cpu()],
  }
  if masques:
    prediction["masks"] = (
      r.masks.data.bool().cpu()
      if r.masks is not None
      else torch.zeros((0, *r.orig_shape), dtype=torch.bool)
    )
  return prediction


modele_seg = YOLO("yolo26n-seg.pt")
r = modele_seg.predict(chemin(ID), device=DEVICE, retina_masks=True, verbose=False)[0]
plt.figure(figsize=(8, 6))
plt.imshow(r.plot()[..., ::-1])
plt.axis("off")
plt.show()
metrique = MeanAveragePrecision(iou_type=("bbox", "segm"))
debut = time.perf_counter()
for i in sous_ensemble:
  p = predire(modele_seg, i, masques=True, conf=0.001, max_det=100, retina_masks=True)
  metrique.update([p], [cibles(i, masques=True)])
print(f"{1000 * (time.perf_counter() - debut) / len(sous_ensemble):.0f} ms par image")
res = metrique.compute()
print({cle: round(float(res[cle]), 3) for cle in ("bbox_map", "segm_map")})
# La mAP des masques est plus basse que celle des boîtes : un masque recouvre moins
# bien son objet qu'une boîte, sur les contours fins (bras, pattes, rayons d'une
# roue) et les objets en partie cachés, et l'IoU de deux masques pénalise chaque
# pixel de contour. Le temps comprend ici l'agrandissement des masques à la taille de
# l'image et la lecture des masques annotés.

## 10. Mask R-CNN

Chargez Mask R-CNN de torchvision (`maskrcnn_resnet50_fpn_v2`, avec les poids `MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT`, appris sur COCO), sur `DEVICE`, en mode évaluation, avec un seuil de score aussi bas que celui de YOLO : `box_score_thresh=0.001` (0,05 par défaut, qui ferait baisser sa mAP comme à l'exercice 6). Ses étiquettes sont déjà les identifiants de COCO. Sur les 25 premières images, évaluez-le comme YOLO26n-seg à l'exercice 9 (boîtes et masques), puis évaluez YOLO26n-seg sur ces mêmes 25 images, et comparez la mAP des masques et le temps par image. Combien de paramètres ont les deux modèles ?

Sur 25 images, la mAP varie beaucoup d'un échantillon à l'autre : comparez les deux modèles sur les mêmes images, et ne retenez que les grands écarts.

**Documentation utile**

- https://github.com/pytorch/vision/blob/main/torchvision/models/detection/mask_rcnn.py

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `poids.transforms()` convertit une image de `decode_image` (entiers `uint8`, `C × H × W`) en flottants entre 0 et 1, ce qu'attend le modèle.
- `sortie["masks"]` a la forme `(N, 1, H, W)`, des probabilités : `sortie["masks"][:, 0] > 0.5` donne les masques.
- La documentation de la classe `MaskRCNN`, au début de `mask_rcnn.py`, liste ses paramètres, que `maskrcnn_resnet50_fpn_v2` lui passe : `box_score_thresh`, `box_detections_per_img` (100 par défaut, comme `max_det`).
- `with torch.no_grad():` autour de l'appel ; `.cpu()` sur chaque tenseur de la sortie.
- `sum(p.numel() for p in m.parameters())` compte les paramètres d'un modèle PyTorch ; `modele_seg.model` est le modèle PyTorch de YOLO.

```python
poids = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
mask_rcnn = (
  maskrcnn_resnet50_fpn_v2(weights=poids, box_score_thresh=...).to(DEVICE).eval()
)


def predire_mask_rcnn(id_image):
  image = poids.transforms()(decode_image(chemin(id_image))).to(DEVICE)
  with torch.no_grad():
    sortie = mask_rcnn([image])[0]
  return {
    "boxes": sortie["boxes"].cpu(),
    "scores": ...,
    "labels": ...,
    "masks": ...,
  }


images = ids_images[:25]
for nom, fonction in {
  "Mask R-CNN": predire_mask_rcnn,
  "YOLO26n-seg": lambda i: predire(
    modele_seg, i, masques=True, conf=..., max_det=..., retina_masks=...
  ),
}.items():
  fonction(images[0])
  metrique = MeanAveragePrecision(iou_type=("bbox", "segm"))
  debut = time.perf_counter()
  for i in images:
    metrique.update([fonction(i)], [cibles(i, masques=True)])
  ...
```

### Solution

In [ ]:
poids = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
mask_rcnn = (
  maskrcnn_resnet50_fpn_v2(weights=poids, box_score_thresh=0.001).to(DEVICE).eval()
)


def predire_mask_rcnn(id_image):
  image = poids.transforms()(decode_image(chemin(id_image))).to(DEVICE)
  with torch.no_grad():
    sortie = mask_rcnn([image])[0]
  return {
    "boxes": sortie["boxes"].cpu(),
    "scores": sortie["scores"].cpu(),
    "labels": sortie["labels"].cpu(),
    "masks": (sortie["masks"][:, 0] > 0.5).cpu(),
  }


images = ids_images[:25]
for nom, fonction in {
  "Mask R-CNN": predire_mask_rcnn,
  "YOLO26n-seg": lambda i: predire(
    modele_seg, i, masques=True, conf=0.001, max_det=100, retina_masks=True
  ),
}.items():
  fonction(images[0])
  metrique = MeanAveragePrecision(iou_type=("bbox", "segm"))
  debut = time.perf_counter()
  for i in images:
    metrique.update([fonction(i)], [cibles(i, masques=True)])
  duree = 1000 * (time.perf_counter() - debut) / len(images)
  res = metrique.compute()
  print(nom, f"{duree:.0f} ms par image", round(float(res["segm_map"]), 3))
for nom, m in {"Mask R-CNN": mask_rcnn, "YOLO26n-seg": modele_seg.model}.items():
  print(nom, f"{sum(p.numel() for p in m.parameters()) / 1e6:.1f} M de paramètres")
# Mask R-CNN a environ quinze fois plus de paramètres que YOLO26n-seg, et prend une
# dizaine de secondes par image sans GPU, contre un quart de seconde. Ses masques sont
# plus précis (une mAP de 0,53 contre 0,42 sur ces 25 images) : sa tête de masque
# travaille sur chaque région découpée par RoIAlign. Le choix suit le même compromis
# qu'à l'exercice 8 : précision ou vitesse.